 在LangChain中，我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色（Role）创建了对应的BaseMessage的子类：
- SystemMessage：role是system，代表系统消息，用于设定模型角色和交互背景
- HumanMessage：role是user，代表用户输入的消息
- AIMessage：role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据
- ToolMessage：role是tool，代表工具调用时产生的结果

In [1]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv()

@tool
def get_weather(location: str) -> str:
    """获取指定城市的天气信息"""
    # 这里可以调用实际的天气API来获取数据
    return f"当前{location}的天气是晴天，温度为25°C。"
#创建Agent
agent = create_agent(model="deepseek-chat",tools= [get_weather])
#调用Agent，发送消息
# response = agent.invoke({
#     "messages": [
#         {"role":"system","content":"你是一个有热心的AI助手。"},
#         {"role": "user", "content": "你好，我是李华。"},
#         {"role": "assistant", "content": "你好，李华！很高兴见到你。请问有什么我可以帮你的吗？"},
#         {"role": "user", "content": "请问你能帮我查一下北京的天气吗？"}
#     ]
# }
response = agent.invoke({
    "messages": [
        SystemMessage(content="你是一个有热心的AI助手。"),
        HumanMessage(content="你好，我是李华。"),
        AIMessage(content="你好，李华！很高兴见到你。请问有什么我可以帮你的吗？"),
        HumanMessage(content="请问你能帮我查一下北京的天气吗？")
    ]
}
)
#打印响应结果
print(response)

{'messages': [SystemMessage(content='你是一个有热心的AI助手。', additional_kwargs={}, response_metadata={}, id='ade5ad75-5e9c-4bf0-8cdc-4a887dac8398'), HumanMessage(content='你好，我是李华。', additional_kwargs={}, response_metadata={}, id='1aaa2ed4-b9d2-4701-8e5f-011b3e2f6d3c'), AIMessage(content='你好，李华！很高兴见到你。请问有什么我可以帮你的吗？', additional_kwargs={}, response_metadata={}, id='d4267e25-97b8-4141-8723-e3f22a05319f', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='请问你能帮我查一下北京的天气吗？', additional_kwargs={}, response_metadata={}, id='a469bedb-046e-4a12-99ce-24a14773e583'), AIMessage(content='好的，我这就帮你查一下北京的天气。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 306, 'total_tokens': 353, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 306}, 'model_provider':

In [2]:
for message in response["messages"]:
    message.pretty_print()

================================ System Message ================================

你是一个有热心的AI助手。
================================ Human Message =================================

你好，我是李华。
================================== Ai Message ==================================

你好，李华！很高兴见到你。请问有什么我可以帮你的吗？
================================ Human Message =================================

请问你能帮我查一下北京的天气吗？
================================== Ai Message ==================================

好的，我这就帮你查一下北京的天气。
Tool Calls:
  get_weather (call_00_wYU7QOXbmpBS38LKVhpU2895)
 Call ID: call_00_wYU7QOXbmpBS38LKVhpU2895
  Args:
    location: 北京
================================= Tool Message =================================
Name: get_weather

当前北京的天气是晴天，温度为25°C。
================================== Ai Message ==================================

李华，北京现在的天气是晴天，气温25°C，挺舒适宜人的。出门的话注意防晒就好～还有什么需要我帮忙的吗？


In [4]:
import os
from langchain.chat_models import init_chat_model

base_url = os.getenv("EVAL_LLM_BASE_URL")
api_key = os.getenv("EVAL_LLM_API_KEY")

model = init_chat_model(model="Qwen/Qwen3.6-35B-A3B",
                        model_provider="openai",
                        base_url=base_url,
                        api_key=api_key
)

In [5]:
agent = create_agent(model=model)
#在线图片消息
message = HumanMessage(
    content=[
        {"type": "image",
         "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
        {"type": "text", "text": "这些图描绘了什么内容？"}
    ])

In [7]:
response = agent.invoke({
    "messages": [
        SystemMessage(content="你是一个有热心的AI助手。"),
        message
    ]
}
)
print(response)

{'messages': [SystemMessage(content='你是一个有热心的AI助手。', additional_kwargs={}, response_metadata={}, id='54731610-2358-4baa-ad2d-b9ec77f6bdc9'), HumanMessage(content=[{'type': 'image', 'url': 'https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg'}, {'type': 'text', 'text': '这些图描绘了什么内容？'}], additional_kwargs={}, response_metadata={}, id='0d368b38-386c-4d58-9f9e-43326e41a7b2'), AIMessage(content='\n\n这张图片描绘了一个非常温馨和愉快的场景，主要内容包括：\n\n1.  **人物与宠物互动**：\n    *   画面中心是一位年轻女性，她穿着深色格子衬衫和深色裤子，赤脚坐在沙滩上。\n    *   她的宠物是一只浅金色或奶油色的狗（看起来很像拉布拉多犬），狗狗戴着彩色的胸背带。\n    *   狗狗正乖巧地坐着，抬起一只前爪，女人伸出双手正在接住或者抚摸狗掌，这看起来像是在进行“握手”或者“击掌”的游戏。两人都面带微笑，神情非常亲密和快乐。\n\n2.  **环境背景**：\n    *   背景是一片大海，可以看到轻微的波浪涌向岸边。\n    *   地面是沙滩，上面有许多脚印。\n\n3.  **光影氛围**：\n    *   光线非常柔和且温暖，呈现出金黄色调，看起来像是日落时分（或者是日出），阳光从右侧照射过来，给画面营造出一种宁静、浪漫且放松的氛围。\n\n总的来说，这是一张展现人与宠物在美好自然环境中享受时光的照片。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 730, 'prompt_tokens': 2782, 'to

In [8]:
#离线图片消息
from ipywidgets import FileUpload
from IPython.display import display
# 创建文件上传控件
uploader= FileUpload(accept='*', multiple=False)
# 显示上传控件
display(uploader)

FileUpload(value=(), accept='*', description='Upload')

In [9]:
print(uploader.value)

({'name': 'img.png', 'type': 'image/png', 'size': 1372644, 'content': <memory at 0x10bc5f4c0>, 'last_modified': datetime.datetime(2026, 11, 9, 12, 39, 20, 718000, tzinfo=datetime.timezone.utc)},)


In [11]:
import base64
multimodel_question = HumanMessage(
    content=[
        {"type": "image",
         "base64": base64.b64encode(uploader.value[0]['content']).decode('utf-8'),
         "mime_type": "image/jpeg",
        },
        {"type": "text", "text": "这些图描绘了什么内容？"}
    ])
# 4.调用Agent，发送多模态消息
for token, metadata in agent.stream({
    "messages": [multimodel_question]
}, stream_mode="messages"):
    if token.content:
        print(token.content, end="", flush=True)



这张图片描绘了一个非常温馨、宁静的场景，主要包含以下内容：

1.  **人物与宠物互动**：
    *   画面中心是一位年轻女性和一只浅黄色的拉布拉多犬（或者类似的寻回犬）。
    *   女性盘腿或侧坐在沙滩上，穿着格纹衬衫和深色裤子。她面带微笑，表情看起来很放松、开心。
    *   狗狗坐在她对面，身体直立，抬起一只前爪，正与女子伸出手的手掌相对。这看起来像是在进行“握手”或“击掌”的训练互动。

2.  **环境背景**：
    *   场景是在海滩上，地面是细腻的沙子，上面有一些脚印和凹陷。
    *   背景是平静的大海，可以看到海浪正在向岸边涌来。
    *   光线非常温暖且明亮，看起来像是日出或日落时分（黄金时刻）。阳光从右侧照射进来，给女子的头发和周围的环境镀上了一层金色的光晕，营造出一种梦幻、温暖的氛围。

总的来说，这是一张展示人与宠物之间亲密关系、充满快乐和宁静气息的生活照。

所谓本地图片，就是用户上传的图片数据或者本地存在的图片，而不是图片的url地址。我们需要将图片数据转换成base64字符串，然后发送给模型。